# Sanity test on Kaggle: easy arithmetic in the IVP-MCQ format

144 trivial items (`+ - * / **` and `a + b * c`), 3 options, one correct. Letters and the rank
of the correct value are balanced; every no-maths heuristic scores 33.3%.
**Expected: very high accuracy.** If the letter score is low here, the problem is the scoring
format, not the maths.

1. letter-score baseline, both prompts per model
2. 6-order consistency test, the thesis prompt per model
3. **what the models write**: greedy free generation on all 144 items vs. the letter score

**Before running** (right-hand panel, *Session options*):
- *Accelerator*: **GPU T4 x2** (one GPU is used) or GPU P100
- *Internet*: **on** (needed to download the models; requires a phone-verified Kaggle account)
- *Input*: add the dataset made from `kaggle_easy_control_files.zip`

Results are written to `/kaggle/working/Thesis/results/easy_control/` and zipped at the end
as `/kaggle/working/easy_control_results.zip` (download it from the *Output* panel).
About 15 min.

In [ ]:
import os, sys, json, re, math, glob, shlex, shutil, subprocess, datetime
INPUT = os.environ.get('KAGGLE_INPUT_DIR', '/kaggle/input')        # read-only dataset mount
WORK = os.environ.get('KAGGLE_WORK_DIR', '/kaggle/working/Thesis')  # writable, kept as output
found = glob.glob(os.path.join(INPUT, '**', 'baseline_letter_logprobs.py'), recursive=True)
assert found, f'project files not found under {INPUT}: add the dataset as an Input of this notebook'
SRC = os.path.dirname(found[0])
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.chdir(WORK)
for f in ['baseline_letter_logprobs.py', 'option_permutations.py', 'easy_mcq/easy_mcq_test.jsonl', 'ivp_mcq/grade_mcq.py']:
    assert os.path.exists(f), f'missing {f} in the uploaded dataset (keep the folder structure of the zip)'
print('project files copied from', SRC, 'to', WORK)
if INPUT == '/kaggle/input':
    r = subprocess.run('curl -s -o /dev/null -w "%{http_code}" https://huggingface.co', shell=True,
                       capture_output=True, text=True)
    assert r.stdout.strip() in ('200', '301', '302'), 'no internet: turn Internet on in Session options'
    subprocess.run('pip -q install "transformers>=4.51" accelerate', shell=True)
sys.path[:0] = [os.getcwd(), os.path.join(os.getcwd(), 'ivp_mcq')]
import torch, transformers
if INPUT == '/kaggle/input':
    assert torch.cuda.is_available(), 'no GPU: set Accelerator to GPU T4 x2 or P100 in Session options'
print('cwd', os.getcwd(), '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
DATA = 'easy_mcq/easy_mcq_test.jsonl'
OUT = 'results/easy_control'
THESIS_PROMPT = {'qwen2.5-3b-instruct': None, 'phi-4-mini-instruct': 'letter-only'}   # as on IVP-MCQ
BOTH_PROMPTS = [None, 'letter-only']
BATCH_SIZE = 8
GEN_TOKENS = 64

## 1-2. Letter score (both prompts) and the 6-order test (thesis prompt)

In [ ]:
def run(script, sub, model, system):
    cmd = (f'{sys.executable} {script} --model {model} --data {DATA} --batch-size {BATCH_SIZE} '
           f'--outdir {OUT}/{sub}' + (f' --system {shlex.quote(system)}' if system else ''))
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if 'acc=' in l]
    if r.returncode or not line:
        raise RuntimeError('run failed:\n' + '\n'.join(r.stderr.strip().splitlines()[-10:]))
    print('  ', line[-1].split('  -> ')[0], flush=True)
    return line[-1].split('  -> ')[1].strip()

runs = {}
for m in THESIS_PROMPT:
    for s in BOTH_PROMPTS:
        runs[(m, s, 'baseline')] = run('baseline_letter_logprobs.py', 'baseline', m, s)
    runs[(m, THESIS_PROMPT[m], 'perm')] = run('option_permutations.py', 'option_permutations', m, THESIS_PROMPT[m])

In [ ]:
base_dirs = ' '.join(f'"{d}/summary.json"' for (m, s, k), d in runs.items() if k == 'baseline')
perm_dirs = ' '.join(f'"{d}/summary.json"' for (m, s, k), d in runs.items() if k == 'perm')
subprocess.run(f'{sys.executable} baseline_letter_logprobs.py --table {base_dirs}', shell=True)
subprocess.run(f'{sys.executable} option_permutations.py --table {perm_dirs}', shell=True)

## 3. What the models write
Greedy generation with the thesis prompt. The letter is read from the text: "answer is X", else the last
`A)`/`B)`/`C)`; if no letter is written, a number that matches exactly one option counts as that option.

In [ ]:
from baseline_letter_logprobs import MODELS, TEMPLATE_KWARGS, SYSTEM_PRESETS, build_text, load_model
from grade_mcq import load
items = load(DATA)

def read_letter(text, it):
    m = re.findall(r'answer is[:\s]*\**\(?([ABC])\b', text, re.I) or re.findall(r'(?<![A-Za-z])([ABC])\)', text)
    if m:
        return m[-1].upper(), 'letter'
    if re.fullmatch(r'\s*\(?[ABC]\)?\.?\s*', text):
        return text.strip().strip('().')[0], 'letter'
    nums = set(re.findall(r'-?\d+', text))
    hit = [o['letter'] for o in it['options'] if o['text'] in nums]
    return (hit[0], 'number') if len(hit) == 1 else (None, 'none')

gen_summary = {}
for m, sysm in THESIS_PROMPT.items():
    hf, fmt = MODELS[m]
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    tok, model = load_model(m, torch.float16 if dev == 'cuda' else torch.float32, dev)
    system = SYSTEM_PRESETS.get(sysm, sysm)
    base = {r['id']: r for r in map(json.loads, open(f"{runs[(m, sysm, 'baseline')]}/predictions.jsonl"))}
    rows = []
    for k, it in enumerate(items):
        t, _ = build_text(tok, it['prompt'], fmt, None, system, TEMPLATE_KWARGS.get(m))
        inp = tok(t, return_tensors='pt', add_special_tokens=False).to(dev)
        with torch.no_grad():
            g = model.generate(**inp, max_new_tokens=GEN_TOKENS, do_sample=False,
                               pad_token_id=tok.pad_token_id or tok.eos_token_id)
        text = tok.decode(g[0, inp['input_ids'].shape[1]:], skip_special_tokens=True)
        L, how = read_letter(text, it)
        rows.append({'id': it['id'], 'category': it['category'], 'gold': it['answer'],
                     'letter_score_pred': base[it['id']]['pred'], 'written': L, 'read_by': how,
                     'written_correct': L == it['answer'], 'text': text})
        print(f'\r{m}: {k + 1}/{len(items)}', end='', flush=True)
    print()
    n = len(rows)
    cats = sorted({r['category'] for r in rows})
    s = {'n': n,
         'letter_score_acc': sum(r['letter_score_pred'] == r['gold'] for r in rows) / n,
         'written_acc': sum(r['written_correct'] for r in rows) / n,
         'no_answer_found': sum(r['written'] is None for r in rows),
         'read_by_number': sum(r['read_by'] == 'number' for r in rows),
         'written_equals_letter_score': sum(r['written'] == r['letter_score_pred'] for r in rows) / n,
         'starts_with_letter': sum(bool(re.match(r'\s*\(?[ABC]\b', r['text'])) for r in rows) / n,
         'per_category_written_acc': {c: sum(r['written_correct'] for r in rows if r['category'] == c)
                                      / sum(r['category'] == c for r in rows) for c in cats}}
    gen_summary[m] = s
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    d = f"{OUT}/generation/{stamp}_{m}{'_sys' if sysm else ''}"
    os.makedirs(d, exist_ok=False)
    with open(f'{d}/generations.jsonl', 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')
    json.dump({'meta': {'model': hf, 'system': system, 'data': DATA, 'max_new_tokens': GEN_TOKENS,
                        'greedy': True, 'transformers': transformers.__version__}, **s},
              open(f'{d}/summary.json', 'w'), indent=2)
    print(json.dumps(s, indent=1))
    for r in rows[:3] + [r for r in rows if not r['written_correct']][:3]:
        print('-' * 80)
        print(r['id'], r['category'], 'gold', r['gold'], '| letter score', r['letter_score_pred'],
              '| written', r['written'], f"({r['read_by']})")
        print(r['text'])
    del model
    if dev == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
print(f"{'model':24s}{'letter-score acc':>18s}{'written acc':>14s}{'agree':>8s}{'starts with letter':>20s}")
for m, s in gen_summary.items():
    print(f"{m:24s}{100 * s['letter_score_acc']:17.1f}%{100 * s['written_acc']:13.1f}%"
          f"{100 * s['written_equals_letter_score']:7.1f}%{100 * s['starts_with_letter']:19.1f}%")

## Download the results

In [ ]:
zip_base = os.path.join(os.path.dirname(WORK), 'easy_control_results')
path = shutil.make_archive(zip_base, 'zip', root_dir=WORK, base_dir='results/easy_control')
print('wrote', path, f'({os.path.getsize(path) / 1e6:.1f} MB) - download it from the Output panel on the right')